In [0]:
WITH category_sales AS (
    SELECT
        p.product_category,
        SUM(f.price) AS category_sales

    FROM `e-commerce`.ecommerce_gold.fact_order_items f

    JOIN `e-commerce`.ecommerce_gold.dim_products p
        ON f.product_sk = p.product_sk

    GROUP BY p.product_category
),

ranked_categories AS (
    SELECT
        *,
        DENSE_RANK() OVER (
            ORDER BY category_sales DESC
        ) AS category_rank

    FROM category_sales
)

SELECT
    ROUND(
        100.0 * SUM(
            CASE
                WHEN category_rank <= 5
                THEN category_sales
                ELSE 0
            END
        ) / NULLIF(SUM(category_sales), 0),
        2
    ) AS top_5_category_contribution_pct

FROM ranked_categories;